# PCB Isolation Design Walkthrough: IEC 60664-1 Clearance & Creepage

This notebook works through a real clearance/creepage sizing problem for a
motor-drive PCB, following the method described in Texas Instruments'
application note *"Circuit Board Insulation Design According to IEC60664 for
Motor Drive Application"* (SLUAAR5, August 2023).

It rebuilds the calculation in Python using small, composable classes for the
**operating environment**, the **voltage domains** on a board, the
**insulation barriers** between them, and a **calculator** that turns those
inputs into clearance/creepage distances and, ultimately, PCB design rules.

**What's new in this version:** the original notebook's calculator only knew
three hand-picked reference points each for clearance and creepage -- enough
to reproduce the white paper's own numbers, but not much else. This version
instead parses the *complete* excerpted IEC 60664-1 Annex F tables straight
out of `IsolationGuide.md` (Tables F.1, F.2, F.4, F.5 and F.10, plus the CTI
material-group table and the pollution-degree/OVC reference tables) and
builds a general-purpose calculator on top of them, so the same code can size
a barrier at *any* voltage, pollution degree, material group, or altitude
those tables cover -- not just the specific numbers TI happened to publish.

> **Safety note:** this notebook is a *teaching example*, not a certified
> design tool. `IsolationGuide.md`'s tables are themselves a hand-transcribed
> excerpt of IEC 60664-1 Annex F -- complete for the rows/columns it
> includes, but not a substitute for the actual standard (which this
> notebook cannot reproduce for copyright reasons). A real product must
> pull the full standard (and, for drives, IEC 61800-5-1) and be reviewed by
> a qualified safety engineer. Anywhere the transcribed tables are
> ambiguous or look inconsistent, this notebook says so explicitly rather
> than guessing -- **see section 13 and the "known caveats" table in the
> final section** for the specific spots worth double-checking.

**Requires:** `IsolationGuide.md` in the same folder as this notebook --
every table below is parsed from it directly, so there's exactly one place
to update if you extend it with more of the standard later.

## 1. Two different failure modes, two different distances

IEC 60664-1 asks for two independent spacings between any two conductors at
different potentials:

- **Clearance** - the shortest straight-line distance *through air*. It
  protects against a fast breakdown (arcing) caused by voltage spikes and
  transients, so it's sized from the **peak/impulse voltage**, including the
  overvoltage category (OVC).
- **Creepage** - the shortest path *along an insulating surface* between the
  same two conductors. It protects against a slow failure mode (surface
  tracking) caused by dirt, humidity and contamination over time, so it's
  sized from the **RMS working voltage**, the pollution degree, and the
  insulating material's comparative tracking index (its "material group").

Because creepage is measured along a surface between the same two points,
it's always geometrically at least as long as clearance - but the two
values are still calculated independently, and a real layout has to respect
both.

The insulation itself is also classified by *how much protection* it has to
provide:

| Type | Purpose |
|---|---|
| Functional | lets the circuit work; no shock protection |
| Basic | single layer of shock protection |
| Double | a second, independent basic layer added in case the first fails |
| Reinforced | a single barrier built to be equivalent to double insulation |

## 2. The example system

We'll use the same worked example as the white paper: an industrial AC motor
drive.

- Class I equipment, metal cabinet, chassis bonded to protective earth (PE)
- 3-phase, 220/380 V wye mains supply, overvoltage category III
- Industrial pollution degree 2 environment
- Galvanically isolated HMI (human-machine interface) required
- "Hot-side" MCU - the microcontroller and gate-drive circuitry sit on the
  same (non-isolated) potential as the power stage
- Installation altitude below 2000 m

The power stage is a standard 3-phase rectifier -> DC bus -> 3-phase
inverter, with the MCU/gate-driver circuitry referenced to the power stage's
local ground, and a small isolated supply feeding a user-facing HMI on the
"cold" side. We'll group the board into five logical **voltage domains**,
matching the block labels A-W used in the white paper's schematic:

| Domain | Blocks | What it is |
|---|---|---|
| `MAINS` | A, B, C | 3-phase 380 Vac line input, upstream of the rectifier |
| `DC_BUS` | D, E, F | Rectified DC bus and hot-side control/gate-drive circuitry |
| `EARTH` | G | Protective earth (PE) |
| `COLD` | H | Isolated ("cold"/user) side reference, after the isolated aux supply |
| `MOTOR` | U, V, W | Inverter output phases to the motor |

Every pair of domains needs its own clearance/creepage spec, because both
the insulation type (functional/basic/reinforced) and the working voltage
across a barrier depend on which two domains you're looking at.

In [1]:
import re
import math
from pathlib import Path
from dataclasses import dataclass
from enum import Enum

import pandas as pd

pd.set_option("display.width", 120)

DATA_FILE = Path("IsolationGuide.md")

## 3. Reference data: pollution degree

Pollution degree characterizes the *micro-environment* right around the
insulation -- it's not the same thing as how clean the factory floor is, and
it directly changes both clearance and creepage requirements. Parsed
straight from the notes:

In [2]:
class PollutionDegree(Enum):
    PD1 = 1
    PD2 = 2
    PD3 = 3
    PD4 = 4


def parse_markdown_table(md_text: str, heading_contains: str, text_cols=frozenset()) -> pd.DataFrame:
    '''
    Find a line in `md_text` containing `heading_contains`, then parse the
    next markdown pipe-table that follows it into a DataFrame. `text_cols`
    are column *positions* to keep as raw strings (e.g. a "120 to 240" or
    "380, 400" style cell) instead of trying to coerce them to a float.
    '''
    lines = md_text.splitlines()
    start = next((i for i, line in enumerate(lines) if heading_contains in line), None)
    if start is None:
        raise ValueError(f"No heading containing {heading_contains!r} found in the notes file")

    j = start + 1
    while j < len(lines) and not lines[j].strip().startswith("|"):
        j += 1
    table_lines = []
    while j < len(lines) and lines[j].strip().startswith("|"):
        table_lines.append(lines[j])
        j += 1
    if not table_lines:
        raise ValueError(f"No table found after heading {heading_contains!r}")

    def split_row(row):
        row = row.strip().strip("|")
        return [c.strip() for c in row.split("|")]

    header = split_row(table_lines[0])
    data_rows = [split_row(r) for r in table_lines[2:]]  # row 1 is the --- separator

    def clean_cell(cell: str, as_text: bool):
        cell = cell.strip()
        if cell in ("", "—", "-", "–"):
            return None
        if as_text or not re.match(r"^[0-9]", cell):
            return cell  # descriptive text, or a footnoted/ranged cell we keep verbatim
        tokens = cell.split()
        first = tokens[0]
        if first.endswith(","):
            num_str = first[:-1]  # "380," "400" -> a comma-separated list, keep just the first value
        else:
            num_str, idx = first, 1
            # European thousands separator: a bare space then exactly 3 digits, e.g. "1 000"
            while idx < len(tokens) and "," not in num_str and re.match(r"^[0-9]{3}$", tokens[idx]):
                num_str += tokens[idx]
                idx += 1
        num_str = num_str.replace(",", ".")
        try:
            return float(num_str)
        except ValueError:
            return cell

    rows = [[clean_cell(c, ci in text_cols) for ci, c in enumerate(row)] for row in data_rows]
    return pd.DataFrame(rows, columns=header)


_md_text = DATA_FILE.read_text(encoding="utf-8") if DATA_FILE.exists() else None
if _md_text is None:
    raise FileNotFoundError(
        f"Couldn't find {DATA_FILE.resolve()}. Keep IsolationGuide.md next to this notebook -- "
        "every reference table below is parsed from it directly."
    )

pollution_degree_table = parse_markdown_table(_md_text, "### Pollution Degree", text_cols={0, 1, 2})
pollution_degree_table

,Pollution Degree,Micro-Environment,Example
0,1,"No pollution or only dry, non-conductive pollu...","Inside air-conditioned labs, under protective ..."
1,2,"Normally, only non-conductive pollution occurs...","Inside standard electrical enclosures, control..."
2,3,"Conductive pollution occurs, or dry non-conduc...",Electrical equipment in industrial environment...
3,4,Continuous or persistent conductivity is gener...,Outdoor electrical equipment or heavy industri...


## 4. Reference data: overvoltage category (OVC)

OVC describes how exposed a part of the circuit is to transient overvoltages
from the mains -- equipment closer to the service entrance sees higher,
less-damped transients and needs a higher OVC rating.

In [3]:
class OvervoltageCategory(Enum):
    OVC_I = "I"
    OVC_II = "II"
    OVC_III = "III"
    OVC_IV = "IV"


overvoltage_category_table = parse_markdown_table(_md_text, "### Overvoltage Category", text_cols={0, 1, 2})
overvoltage_category_table

,Category,Characteristics,Examples
0,Category I (OVC I),applies to equipment connected to a circuit wh...,Inside the electrical circuit.
1,Category II (OVC II),applies between the circuits not directly supp...,"Appliances, portable tools and other plug-conn..."
2,Category III (OVC III),applies to the circuits directly supplied by m...,Downstream of and including the main distribut...
3,Category IV (OVC IV),applies to equipment permanently connected at ...,Upstream of the main distribution board like e...


## 5. Reference data: comparative tracking index (CTI) & material groups

Creepage failure is a tracking process: contamination + moisture + voltage
slowly carbonize a conductive path across an insulator's surface. The
Comparative Tracking Index (CTI) is a standardized measure of how resistant
a material is to that, and IEC 60664-1 buckets materials into four groups
by CTI range -- a PCB's soldermask/laminate combination determines which
group you get to use in the creepage tables later on.

In [4]:
class MaterialGroup(Enum):
    I = "I"
    II = "II"
    IIIa = "IIIa"
    IIIb = "IIIb"

    @classmethod
    def from_cti(cls, cti_value: float, table: pd.DataFrame = None) -> "MaterialGroup":
        table = cti_groups_table if table is None else table
        for _, row in table.iterrows():
            nums = [float(n) for n in re.findall(r"\d+", row["CTI Range"])]
            lo, hi = (nums[0], math.inf) if len(nums) == 1 else (nums[0], nums[1])
            if lo <= cti_value < hi:
                return cls(row["Material Group"])
        raise ValueError(f"No material group covers CTI={cti_value}")


cti_groups_table = parse_markdown_table(_md_text, "Comparative tracking", text_cols={1})
cti_groups_table

,Material Group,CTI Range
0,I,600 V < CTI
1,II,400 V < CTI < 600 V
2,IIIa,175 V < CTI < 400 V
3,IIIb,100 V < CTI < 175 V


In [5]:
# quick sanity check: a CTI of 250 (typical FR-4 soldermask) should land in IIIa
MaterialGroup.from_cti(250)

<MaterialGroup.IIIa: 'IIIa'>

## 6. Insulation types, protection classes, and extra-low voltage

A few more classifications from the notes round out the vocabulary, even
though the calculator itself only directly consumes `InsulationType`:

- **Functional / Basic / Double / Reinforced** insulation (introduced in
  section 1) -- `InsulationType` below.
- **Protection class** -- *Class I* equipment relies on a protective earth
  connection as a backstop if basic insulation fails; *Class II* equipment
  instead relies on double/reinforced insulation and has no protective
  earth. Our worked example is Class I (metal cabinet bonded to PE).
- **Extra-low voltage (ELV)** -- any voltage not exceeding 50 Vac-rms /
  120 Vdc. Circuits that never leave ELV get a lot more design freedom,
  which is why the white paper calls it out explicitly in Step 2 (voltage
  block definition).

In [6]:
class InsulationType(Enum):
    FUNCTIONAL = "functional"
    BASIC = "basic"
    DOUBLE = "double"
    REINFORCED = "reinforced"


class ProtectionClass(Enum):
    CLASS_I = "I"    # relies on protective earth as a backstop
    CLASS_II = "II"  # relies on double/reinforced insulation, no earth


ELV_AC_RMS_V = 50
ELV_DC_V = 120

def is_extra_low_voltage(ac_rms_v: float = 0, dc_v: float = 0) -> bool:
    return ac_rms_v <= ELV_AC_RMS_V and dc_v <= ELV_DC_V

## 7. Modeling the environment

Pollution degree, PCB material group, and installation altitude all shift
the required distances. We capture them once, as an `Environment` object,
so every calculation downstream stays consistent. `on_printed_wiring` flags
whether the creepage path in question runs directly along the bare PCB
surface (Table F.5 has a dedicated, tighter column for that) or across some
other insulating material classified purely by material group.

In [7]:
@dataclass
class Environment:
    pollution_degree: PollutionDegree
    material_group: MaterialGroup = MaterialGroup.IIIa
    altitude_m: float = 2000
    on_printed_wiring: bool = True

    def describe(self) -> str:
        return (
            f"Pollution degree {self.pollution_degree.value}, "
            f"material group {self.material_group.value}, "
            f"altitude {self.altitude_m:.0f} m, "
            f"{'on bare PCB surface' if self.on_printed_wiring else 'off-PWB / general material'}"
        )


env = Environment(pollution_degree=PollutionDegree.PD2, material_group=MaterialGroup.IIIa,
                   altitude_m=2000, on_printed_wiring=False)
print(env.describe())

Pollution degree 2, material group IIIa, altitude 2000 m, off-PWB / general material


## 8. Modeling the voltage domains

Each `VoltageBlock` is just a labeled node on the board. We don't attach a
working voltage to the block itself, because the *relevant* working voltage
between two domains depends on which pair you're evaluating - mains-to-earth
sees a different working voltage than mains-to-motor-output, for example.

In [8]:
@dataclass
class VoltageBlock:
    name: str
    description: str


MAINS = VoltageBlock("MAINS", "3x380 Vac wye mains input (A/B/C phases)")
DC_BUS = VoltageBlock("DC_BUS", "Rectified DC bus + hot-side MCU/gate-drive circuitry (D/E/F)")
EARTH = VoltageBlock("EARTH", "Protective earth / chassis (G)")
COLD = VoltageBlock("COLD", "Isolated 'cold'/user-side reference, after the aux supply (H)")
MOTOR = VoltageBlock("MOTOR", "Inverter output phases to the motor (U/V/W)")

domains = [MAINS, DC_BUS, EARTH, COLD, MOTOR]
for d in domains:
    print(f"{d.name:8s} - {d.description}")

MAINS    - 3x380 Vac wye mains input (A/B/C phases)
DC_BUS   - Rectified DC bus + hot-side MCU/gate-drive circuitry (D/E/F)
EARTH    - Protective earth / chassis (G)
COLD     - Isolated 'cold'/user-side reference, after the aux supply (H)
MOTOR    - Inverter output phases to the motor (U/V/W)


## 9. Modeling a barrier between two domains

An `InsulationBarrier` is the actual thing we need a spacing for: a specific
pair of voltage domains, the insulation type required between them, the
overvoltage category, and the RMS working voltage used for creepage.

For clearance, you can supply the **impulse (peak) voltage directly** (if
it's already known from simulation/measurement, as in the white paper's own
Step 4), *or* leave it out and supply `nominal_line_to_neutral_v` instead --
the calculator will then derive it from Table F.1 in section 12. Either way,
the working voltage for creepage is a real engineering input, not something
this notebook derives on its own: per the white paper's Step 4, "the
working voltage value is done by simulation and calculation."

In [9]:
@dataclass
class InsulationBarrier:
    domain_a: VoltageBlock
    domain_b: VoltageBlock
    insulation_type: InsulationType
    ovc: OvervoltageCategory
    working_voltage_rms_v: float
    impulse_voltage_v: float = None          # known directly, OR:
    nominal_line_to_neutral_v: float = None  # ... derive it from Table F.1

    @property
    def label(self) -> str:
        return f"{self.domain_a.name} - {self.domain_b.name}"

    def __repr__(self):
        return f"<Barrier {self.label}: {self.insulation_type.value}, OVC {self.ovc.value}>"

## 10. Loading the full IEC 60664-1 Annex F tables

This is the main upgrade over the original notebook. Instead of three
hand-picked points, `IEC60664Tables` parses every relevant table straight
out of `IsolationGuide.md`:

| Table | Contents | Used for |
|---|---|---|
| F.1 | Rated impulse withstand voltage by nominal line-to-neutral voltage and OVC | Deriving an impulse (peak) voltage from a nominal mains voltage |
| F.4 | Rationalized line-to-line / line-to-earth voltages for 3-phase systems | Converting a nominal mains voltage into the line-to-neutral figure Table F.1 wants |
| F.2 | Clearance vs. required impulse withstand voltage, by case (in/homogeneous field) and pollution degree | Clearance |
| F.5 | Creepage vs. RMS working voltage, by pollution degree and material group | Creepage |
| F.10 | Altitude correction factor for installations *below* 2000 m | Optional credit for a low-altitude installation |
| CTI table | Material group vs. CTI range | Classifying a PCB material |

Table F.2 has some blank cells at low voltages for pollution degrees 2 and
3 -- these aren't missing data so much as the table's way of saying "the
pollution-degree floor still applies here, unchanged from the nearest
tabulated value." We reconstruct that floor with a forward/backward-fill
down each column, so every filled-in cell is a real number copied from
elsewhere in that same column, never invented.

In [10]:
class IEC60664Tables:
    def __init__(self, path: Path = DATA_FILE):
        self.path = Path(path)
        text = self.path.read_text(encoding="utf-8")

        self.rated_impulse_voltage = (
            parse_markdown_table(text, "Table F.1 -", text_cols={0, 1})
            .sort_values("Voltage line to neutral derived from nominal voltages AC or DC (V)")
            .reset_index(drop=True)
        )
        self.nominal_voltage_map = parse_markdown_table(text, "Table F.4 -", text_cols={0})
        self.clearance = self._prepare_clearance_table(parse_markdown_table(text, "Table F.2 -"))
        self.creepage = (
            parse_markdown_table(text, "Table F.5 -").sort_values("Voltage RMS (V)").reset_index(drop=True)
        )
        self.low_altitude_factor = (
            parse_markdown_table(text, "Table F.10 -").sort_values("Altitude (m)").reset_index(drop=True)
        )

    @staticmethod
    def _prepare_clearance_table(df: pd.DataFrame) -> pd.DataFrame:
        df = df.sort_values("Required impulse withstand voltage (kV)").reset_index(drop=True)
        value_cols = [c for c in df.columns if c != "Required impulse withstand voltage (kV)"]
        df[value_cols] = df[value_cols].ffill().bfill()
        return df


tables = IEC60664Tables()
print("Clearance table (Table F.2, gaps reconstructed as pollution-degree floors):")
tables.clearance

Clearance table (Table F.2, gaps reconstructed as pollution-degree floors):


,Required impulse withstand voltage (kV),Case A (Inhomogeneous field) - Pollution degree 1 (mm),Case A - Pollution degree 2 (mm),Case A - Pollution degree 3 (mm),Case B (Homogeneous field) - Pollution degree 1 (mm),Case B - Pollution degree 2 (mm),Case B - Pollution degree 3 (mm)
0,0.33,0.01,0.20,0.8,0.01,0.20,0.8
1,0.40,0.02,0.20,0.8,0.02,0.20,0.8
2,0.50,0.04,0.20,0.8,0.04,0.20,0.8
3,0.60,0.06,0.20,0.8,0.06,0.20,0.8
4,0.80,0.10,0.20,0.8,0.10,0.20,0.8
5,1.00,0.15,0.25,0.8,0.15,0.20,0.8
6,1.20,0.25,0.25,0.8,0.20,0.20,0.8
7,1.50,0.50,0.50,0.8,0.30,0.30,0.8
8,2.00,1.00,1.00,1.0,0.45,0.45,0.8
9,2.50,1.50,1.50,1.5,0.60,0.60,0.8


In [11]:
print("Creepage table (Table F.5) -- first and last few rows:")
pd.concat([tables.creepage.head(3), tables.creepage.tail(3)])

Creepage table (Table F.5) -- first and last few rows:


,Voltage RMS (V),Pollution degree 1: Printed wiring material (mm),Pollution degree 1: All material groups (mm),Pollution degree 2: Printed wiring material (mm),Pollution degree 2: All material groups except IIIb (mm),Pollution degree 2: Material group II (mm),Pollution degree 2: Material group I (mm),Pollution degree 3: Material group III (mm),Pollution degree 3: Material group II (mm),Pollution degree 3: Material group I (mm)
0,10.0,0.025,0.08,0.04,0.40,0.40,0.40,1.00,1.00,1.00
1,12.5,0.025,0.09,0.04,0.42,0.42,0.42,1.05,1.05,1.05
2,16.0,0.025,0.10,0.04,0.45,0.45,0.45,1.10,1.10,1.10
36,40000.0,NaN,160.00,NaN,280.00,200.00,400.00,500.00,600.00,630.00
37,50000.0,NaN,200.00,NaN,360.00,250.00,500.00,630.00,NaN,NaN
38,63000.0,NaN,250.00,NaN,450.00,320.00,630.00,NaN,NaN,NaN


## 11. The general clearance/creepage calculator

This replaces the original notebook's toy `IEC60664Calculator` -- same job,
but every lookup now goes against the full tables from section 10 instead
of three hardcoded pairs, so it works for whatever voltage, pollution
degree, material group, or altitude those tables cover.

A few implementation notes:

- **Clearance (Table F.2)** is a *stepped* lookup, not an interpolation: you
  design to the next preferred impulse-voltage value, not a value in
  between. `clearance_mm()` rounds the requested impulse voltage **up** to
  the next tabulated step before reading the column.
- **Creepage (Table F.5)** *is* linearly interpolated between the two
  bracketing RMS-voltage rows, same as the original notebook did.
- **Reinforced insulation** gets double the basic/functional creepage at
  the same working voltage, and -- per the white paper's Step 5 text --
  its clearance is dimensioned "one step higher in the preferred series"
  than basic insulation. `rated_impulse_voltage()` implements that directly:
  for reinforced insulation it takes the *next row down* in Table F.1
  (i.e. the next higher line-to-neutral voltage class) before reading the
  OVC column.
- Every lookup raises a clear `ValueError` rather than silently
  extrapolating if you ask for something outside the transcribed range --
  fabricating an insulation-safety number is worse than admitting the
  notebook doesn't have it.

In [12]:
class IEC60664Calculator:
    _CLEARANCE_COLS = {
        ("A", 1): "Case A (Inhomogeneous field) - Pollution degree 1 (mm)",
        ("A", 2): "Case A - Pollution degree 2 (mm)",
        ("A", 3): "Case A - Pollution degree 3 (mm)",
        ("B", 1): "Case B (Homogeneous field) - Pollution degree 1 (mm)",
        ("B", 2): "Case B - Pollution degree 2 (mm)",
        ("B", 3): "Case B - Pollution degree 3 (mm)",
    }

    # Table A.2 (design-time credit/penalty for installation above 2000 m).
    # IsolationGuide.md only quotes these two reference points in prose
    # (Step 6) -- pull the complete Table A.2 from the standard for other
    # altitudes rather than extrapolating this pair.
    _INSTALL_ALTITUDE_CORRECTION = {2000: 1.00, 4000: 1.29}

    def __init__(self, environment: Environment, tables: IEC60664Tables):
        self.env = environment
        self.tables = tables

    # ---- Table F.1 -> F.4: rated impulse withstand voltage -------------
    def rated_impulse_voltage(self, line_to_neutral_v: float, ovc: OvervoltageCategory,
                               insulation_type: InsulationType = InsulationType.BASIC) -> float:
        '''Round the line-to-neutral working voltage up to the next standard
        step in Table F.1 and read off the rated impulse withstand voltage.
        Reinforced insulation uses the row one step higher (Step 5 of the
        white paper's method).'''
        df = self.tables.rated_impulse_voltage
        key_col = "Voltage line to neutral derived from nominal voltages AC or DC (V)"
        candidates = df[df[key_col] >= line_to_neutral_v]
        if candidates.empty:
            raise ValueError(f"{line_to_neutral_v} V exceeds Table F.1's tabulated range")
        row_idx = candidates.index[0]
        if insulation_type == InsulationType.REINFORCED:
            row_idx += 1
            if row_idx >= len(df):
                raise ValueError("Reinforced insulation needs one row higher than the top of Table F.1")
        col = f"Rated impulse withstand voltage - Overvoltage category {ovc.value} (V)"
        return float(df.loc[row_idx, col])

    def nominal_voltage_to_line_to_earth(self, nominal_three_phase_v: float,
                                          wiring: str = "four_wire_neutral_earthed") -> float:
        '''Table F.4: rationalize a 3-phase nominal mains voltage into the
        line-to-earth voltage Table F.1 expects as its lookup key.'''
        col = {
            "line_to_line": "For insulation line-to-line All systems (V)",
            "four_wire_neutral_earthed": "For insulation line-to-earth Three-phase four-wire systems neutral-earthed (V)",
            "three_wire_unearthed": "For insulation line-to-earth Three-phase three-wire systems unearthed or corner-earthed (V)",
        }[wiring]
        for _, row in self.tables.nominal_voltage_map.iterrows():
            tokens = re.split(",", str(row.iloc[0]))
            values = [float(re.sub(r"[^0-9]", "", t)) for t in tokens if re.search(r"\d", t)]
            if any(abs(v - nominal_three_phase_v) < 1e-6 for v in values):
                val = row[col]
                if val is None or (isinstance(val, float) and math.isnan(val)):
                    raise ValueError(f"Table F.4 has no {wiring!r} entry for {nominal_three_phase_v} V")
                return float(val)
        raise ValueError(f"{nominal_three_phase_v} V isn't one of Table F.4's tabulated nominal voltages")

    # ---- Table F.2: clearance -------------------------------------------
    def clearance_mm(self, impulse_voltage_v: float, field: str = "A") -> float:
        impulse_kv = impulse_voltage_v / 1000.0
        df = self.tables.clearance
        key_col = "Required impulse withstand voltage (kV)"
        candidates = df[df[key_col] >= impulse_kv - 1e-9]
        if candidates.empty:
            raise ValueError(f"{impulse_kv} kV exceeds Table F.2's tabulated range")
        col = self._CLEARANCE_COLS[(field, self.env.pollution_degree.value)]
        return float(candidates.iloc[0][col])

    def install_altitude_correction_factor(self) -> float:
        '''Table A.2 -- multiply clearance by this for installation ABOVE 2000 m.'''
        if self.env.altitude_m <= 2000:
            return 1.0
        try:
            return self._INSTALL_ALTITUDE_CORRECTION[self.env.altitude_m]
        except KeyError:
            raise ValueError(
                f"No Table A.2 reference point at {self.env.altitude_m} m in these notes -- "
                "look up IEC 60664-1 Table A.2 directly."
            )

    def low_altitude_clearance_factor(self, altitude_m: float = None) -> float:
        '''Table F.10 -- optional credit for an installation altitude BELOW
        2000 m (air is denser, so a smaller clearance meets the same
        dielectric strength). Table F.2's own values are already the
        conservative "up to 2000 m" figures, i.e. this factor is 1.0 at
        2000 m and shrinks toward 0.784 at sea level.'''
        altitude_m = self.env.altitude_m if altitude_m is None else altitude_m
        df = self.tables.low_altitude_factor
        x, y = df["Altitude (m)"].to_numpy(), df["Factor kd for distance correction"].to_numpy()
        if altitude_m <= x.min():
            return float(y[0])
        if altitude_m >= x.max():
            return float(y[-1])
        return float(pd.Series(y, index=x).reindex(sorted(set(x) | {altitude_m})).interpolate("index").loc[altitude_m])

    # ---- Table F.5: creepage ---------------------------------------------
    def _creepage_column(self) -> str:
        pd_, mg = self.env.pollution_degree.value, self.env.material_group
        if pd_ == 1:
            return ("Pollution degree 1: Printed wiring material (mm)" if self.env.on_printed_wiring
                    else "Pollution degree 1: All material groups (mm)")
        if pd_ == 2:
            if self.env.on_printed_wiring:
                return "Pollution degree 2: Printed wiring material (mm)"
            if mg == MaterialGroup.I:
                return "Pollution degree 2: Material group I (mm)"
            if mg == MaterialGroup.II:
                return "Pollution degree 2: Material group II (mm)"
            if mg == MaterialGroup.IIIa:
                return "Pollution degree 2: All material groups except IIIb (mm)"
            raise ValueError("Material group IIIb has no dedicated column at PD2 in this excerpt -- consult the standard directly")
        if pd_ == 3:
            return {
                MaterialGroup.I: "Pollution degree 3: Material group I (mm)",
                MaterialGroup.II: "Pollution degree 3: Material group II (mm)",
            }.get(mg, "Pollution degree 3: Material group III (mm)")
        raise ValueError(f"Table F.5 in these notes doesn't cover pollution degree {pd_}")

    def creepage_mm(self, working_voltage_rms_v: float, insulation_type: InsulationType) -> float:
        col = self._creepage_column()
        sub = self.tables.creepage[["Voltage RMS (V)", col]].dropna()
        lower = sub[sub["Voltage RMS (V)"] <= working_voltage_rms_v].tail(1)
        upper = sub[sub["Voltage RMS (V)"] >= working_voltage_rms_v].head(1)
        if lower.empty or upper.empty:
            lo, hi = sub["Voltage RMS (V)"].min(), sub["Voltage RMS (V)"].max()
            raise ValueError(f"{working_voltage_rms_v} V rms is outside {col!r}'s tabulated range ({lo}-{hi} V)")
        v0, c0 = lower.iloc[0]
        v1, c1 = upper.iloc[0]
        basic = c0 if v0 == v1 else (working_voltage_rms_v - v0) * (c1 - c0) / (v1 - v0) + c0
        return 2 * basic if insulation_type in (InsulationType.REINFORCED, InsulationType.DOUBLE) else basic

    # ---- putting it together ---------------------------------------------
    def evaluate(self, barrier: InsulationBarrier) -> dict:
        impulse_v = barrier.impulse_voltage_v
        if impulse_v is None:
            if barrier.nominal_line_to_neutral_v is None:
                raise ValueError(f"{barrier.label}: need impulse_voltage_v or nominal_line_to_neutral_v")
            impulse_v = self.rated_impulse_voltage(barrier.nominal_line_to_neutral_v, barrier.ovc, barrier.insulation_type)
        clearance = round(self.clearance_mm(impulse_v), 3)
        clearance_alt = round(clearance * self.install_altitude_correction_factor(), 3)
        creepage = round(self.creepage_mm(barrier.working_voltage_rms_v, barrier.insulation_type), 3)
        return {
            "barrier": barrier.label,
            "insulation": barrier.insulation_type.value,
            "OVC": barrier.ovc.value,
            "impulse_V": impulse_v,
            "working_V_rms": barrier.working_voltage_rms_v,
            "clearance_mm": clearance,
            "clearance_mm_altitude_corrected": clearance_alt,
            "creepage_mm": creepage,
            "design_spacing_mm": round(max(clearance_alt, creepage), 3),
        }


calc = IEC60664Calculator(env, tables)

## 12. Reproducing the white paper's worked numbers

Same check as before -- but now every number below comes from the full
Table F.2 / F.5, not a 3-point stand-in.

In [13]:
print("Clearance (Table F.2, PD2):")
for label, iv in [("functional, OVC I", 1500), ("basic, OVC III", 4000), ("reinforced, OVC III", 6000)]:
    print(f"  {label:22s} {iv:5d} V impulse -> {calc.clearance_mm(iv)} mm")

print()
print("Impulse voltage auto-derived from nominal voltage (Table F.4 -> F.1),")
print("for the 380 V wye mains supply, OVC III:")
line_to_earth = calc.nominal_voltage_to_line_to_earth(380)
print(f"  Table F.4: 380 V nominal -> {line_to_earth} V line-to-earth (4-wire, neutral-earthed)")
print(f"  Table F.1: basic      -> {calc.rated_impulse_voltage(line_to_earth, OvervoltageCategory.OVC_III, InsulationType.BASIC)} V")
print(f"  Table F.1: reinforced -> {calc.rated_impulse_voltage(line_to_earth, OvervoltageCategory.OVC_III, InsulationType.REINFORCED)} V")
print("  (matches the white paper's own 4000 V / 6000 V exactly)")

print()
print("Creepage (Table F.5, PD2, material group IIIa, off bare PCB surface):")
for wv in (400, 440, 565):
    print(f"  {wv} V rms, basic -> {calc.creepage_mm(wv, InsulationType.BASIC):.3f} mm")
print(f"  400 V rms, reinforced -> {calc.creepage_mm(400, InsulationType.REINFORCED):.3f} mm")

Clearance (Table F.2, PD2):
  functional, OVC I       1500 V impulse -> 0.5 mm
  basic, OVC III          4000 V impulse -> 3.0 mm
  reinforced, OVC III     6000 V impulse -> 5.5 mm

Impulse voltage auto-derived from nominal voltage (Table F.4 -> F.1),
for the 380 V wye mains supply, OVC III:
  Table F.4: 380 V nominal -> 250.0 V line-to-earth (4-wire, neutral-earthed)
  Table F.1: basic      -> 4000.0 V
  Table F.1: reinforced -> 6000.0 V
  (matches the white paper's own 4000 V / 6000 V exactly)

Creepage (Table F.5, PD2, material group IIIa, off bare PCB surface):
  400 V rms, basic -> 2.800 mm
  440 V rms, basic -> 3.120 mm
  565 V rms, basic -> 4.050 mm
  400 V rms, reinforced -> 5.600 mm


## 13. A closer look: where the full table disagrees with the white paper's text

The clearance numbers and the auto-derived impulse voltages match the white
paper exactly. **The creepage numbers don't**, and the full table makes the
mismatch more specific than the original notebook's toy version could:

| Working voltage | White paper text (Step 6) | This notebook, "all groups except IIIb" column |
|---|---|---|
| 400 V | 2.00 mm | 2.80 mm |
| 440 V | 2.20 mm | ~3.12 mm |
| 565 V | 2.80 mm | ~4.05 mm |

Table F.5 actually has *four* PD2 columns (printed-wiring material, "all
groups except IIIb", group II, and group I), and none of them reproduces
2.00 mm at 400 V exactly -- the closest match is the **"Material group II"**
column (2.0 mm at 400 V), not "all groups except IIIb" (2.8 mm), even
though the system is modeled as material group IIIa.

Two independent things are worth flagging here, not just one:

1. **Which PD2 column an IIIa material should use** is genuinely ambiguous
   from the transcribed table alone -- "all groups except IIIb" reads as
   the literal match for "IIIa," but doesn't reproduce the paper's numbers.
2. **Table F.5's group ordering looks inverted.** IEC 60664-1's usual
   convention is that a *better* material (higher CTI, i.e. group I) needs
   *less* creepage than a worse one (group II, then IIIa/b). In this
   transcribed table, at 400 V, group I reads **4.0 mm** -- larger than
   group II's **2.0 mm** -- the opposite of that convention. That pattern
   holds at every voltage step where both columns are defined, which looks
   more like a mislabeling (group I/II swapped, or the whole progression
   reversed) during transcription than a one-off typo.

None of this is a bug in the calculator -- it's doing exactly what the
labeled columns say. It's a reason to **verify the Table F.5 column
headers against IEC 60664-1 directly** before trusting group-specific
creepage numbers for anything beyond this teaching exercise. The rest of
this notebook keeps using the literal column labels as transcribed, and
flags results that came from a questionable column when it matters.

## 14. Applying it across the whole board

One `InsulationBarrier` per domain pair, using the insulation type / OVC /
working-voltage assignments the white paper works out in its Step 3 and
Step 4. Clearance is supplied directly here (matching the paper's own
numbers exactly, including the two reinforced barriers using "one step
higher"); feel free to swap any of these to `nominal_line_to_neutral_v=...`
instead to see the calculator derive it live, as demonstrated in section 12.

In [14]:
barriers = [
    InsulationBarrier(MAINS,  MAINS,  InsulationType.BASIC,      OvervoltageCategory.OVC_III, 400, impulse_voltage_v=4000),
    InsulationBarrier(MAINS,  DC_BUS, InsulationType.BASIC,      OvervoltageCategory.OVC_III, 400, impulse_voltage_v=4000),
    InsulationBarrier(MAINS,  EARTH,  InsulationType.BASIC,      OvervoltageCategory.OVC_III, 400, impulse_voltage_v=4000),
    InsulationBarrier(MAINS,  COLD,   InsulationType.REINFORCED, OvervoltageCategory.OVC_III, 440, impulse_voltage_v=6000),
    InsulationBarrier(MAINS,  MOTOR,  InsulationType.BASIC,      OvervoltageCategory.OVC_III, 440, impulse_voltage_v=4000),
    InsulationBarrier(DC_BUS, DC_BUS, InsulationType.FUNCTIONAL, OvervoltageCategory.OVC_I,   565, impulse_voltage_v=1500),
    InsulationBarrier(DC_BUS, EARTH,  InsulationType.BASIC,      OvervoltageCategory.OVC_III, 400, impulse_voltage_v=4000),
    InsulationBarrier(DC_BUS, COLD,   InsulationType.REINFORCED, OvervoltageCategory.OVC_III, 440, impulse_voltage_v=6000),
    InsulationBarrier(DC_BUS, MOTOR,  InsulationType.FUNCTIONAL, OvervoltageCategory.OVC_I,   565, impulse_voltage_v=1500),
    InsulationBarrier(EARTH,  MOTOR,  InsulationType.BASIC,      OvervoltageCategory.OVC_III, 400, impulse_voltage_v=4000),
    InsulationBarrier(COLD,   MOTOR,  InsulationType.REINFORCED, OvervoltageCategory.OVC_III, 440, impulse_voltage_v=6000),
    InsulationBarrier(MOTOR,  MOTOR,  InsulationType.FUNCTIONAL, OvervoltageCategory.OVC_I,   400, impulse_voltage_v=1500),
]

results = pd.DataFrame([calc.evaluate(b) for b in barriers])
results

,barrier,insulation,OVC,impulse_V,working_V_rms,clearance_mm,clearance_mm_altitude_corrected,creepage_mm,design_spacing_mm
0,MAINS - MAINS,basic,III,4000,400,3.0,3.0,2.80,3.00
1,MAINS - DC_BUS,basic,III,4000,400,3.0,3.0,2.80,3.00
2,MAINS - EARTH,basic,III,4000,400,3.0,3.0,2.80,3.00
3,MAINS - COLD,reinforced,III,6000,440,5.5,5.5,6.24,6.24
4,MAINS - MOTOR,basic,III,4000,440,3.0,3.0,3.12,3.12
5,DC_BUS - DC_BUS,functional,I,1500,565,0.5,0.5,4.05,4.05
6,DC_BUS - EARTH,basic,III,4000,400,3.0,3.0,2.80,3.00
7,DC_BUS - COLD,reinforced,III,6000,440,5.5,5.5,6.24,6.24
8,DC_BUS - MOTOR,functional,I,1500,565,0.5,0.5,4.05,4.05
9,EARTH - MOTOR,basic,III,4000,400,3.0,3.0,2.80,3.00


One barrier is missing from the table above: `EARTH-COLD` (the G-H pair), a
low-voltage control-reference connection at roughly 50 V (clearance) / 10 V
(creepage) -- below Table F.2's tabulated range, but comfortably inside
Table F.5's. We can derive the creepage from the full table directly and
take the clearance from the white paper's own published value (still below
what Table F.2 covers even after the section-10 floor reconstruction).

In [15]:
earth_cold_creepage = round(calc.creepage_mm(10, InsulationType.BASIC), 3)

given_values = pd.DataFrame([{
    "barrier": "EARTH - COLD",
    "insulation": "basic",
    "OVC": "I",
    "impulse_V": None,
    "working_V_rms": 10,
    "clearance_mm": 0.2,
    "clearance_mm_altitude_corrected": 0.2,
    "creepage_mm": earth_cold_creepage,
    "design_spacing_mm": max(0.2, earth_cold_creepage),
}])

results["source"] = "calculated"
given_values["source"] = "clearance taken from SLUAAR5 Table 2-3 (below Table F.2's range); creepage from Table F.5"
all_results = pd.concat([results, given_values], ignore_index=True)
all_results

,barrier,insulation,OVC,impulse_V,working_V_rms,clearance_mm,clearance_mm_altitude_corrected,creepage_mm,design_spacing_mm,source
0,MAINS - MAINS,basic,III,4000,400,3.0,3.0,2.80,3.00,calculated
1,MAINS - DC_BUS,basic,III,4000,400,3.0,3.0,2.80,3.00,calculated
2,MAINS - EARTH,basic,III,4000,400,3.0,3.0,2.80,3.00,calculated
3,MAINS - COLD,reinforced,III,6000,440,5.5,5.5,6.24,6.24,calculated
4,MAINS - MOTOR,basic,III,4000,440,3.0,3.0,3.12,3.12,calculated
5,DC_BUS - DC_BUS,functional,I,1500,565,0.5,0.5,4.05,4.05,calculated
6,DC_BUS - EARTH,basic,III,4000,400,3.0,3.0,2.80,3.00,calculated
7,DC_BUS - COLD,reinforced,III,6000,440,5.5,5.5,6.24,6.24,calculated
8,DC_BUS - MOTOR,functional,I,1500,565,0.5,0.5,4.05,4.05,calculated
9,EARTH - MOTOR,basic,III,4000,400,3.0,3.0,2.80,3.00,calculated


## 15. Altitude correction, both directions

Table F.2's values are already the conservative figures for "up to 2000 m
above sea level." Two separate corrections exist for that baseline:

- **Above 2000 m (Table A.2):** thinner air is a worse insulator, so
  clearance must *increase*. Only two reference points are transcribed in
  the notes (2000 m -> 1.00x, 4000 m -> 1.29x) -- pull the complete table
  for anything else.
- **Below 2000 m (Table F.10):** denser air is a *better* insulator, so a
  design that only needs to work at, say, sea level can optionally take
  credit for a smaller clearance. This factor is 1.0 at 2000 m (matching
  Table F.2's own baseline) and shrinks toward 0.784 at sea level, and can
  be interpolated for any altitude in between.

In [16]:
env_high_alt = Environment(pollution_degree=PollutionDegree.PD2, material_group=MaterialGroup.IIIa, altitude_m=4000)
calc_high_alt = IEC60664Calculator(env_high_alt, tables)

base_clearance = calc.clearance_mm(4000)  # basic insulation, OVC III, at or below 2000 m
factor = calc_high_alt.install_altitude_correction_factor()
print(f"Basic-insulation clearance up to 2000 m: {base_clearance} mm")
print(f"Same barrier at 4000 m (Table A.2):      {base_clearance} mm x {factor} = {base_clearance * factor:.2f} mm")

print()
print("Optional credit for a LOW-altitude installation (Table F.10):")
for alt in (0, 500, 1000, 1500, 2000):
    print(f"  {alt:5.0f} m -> factor {calc.low_altitude_clearance_factor(alt):.3f} "
          f"-> {base_clearance * calc.low_altitude_clearance_factor(alt):.3f} mm")

Basic-insulation clearance up to 2000 m: 3.0 mm
Same barrier at 4000 m (Table A.2):      3.0 mm x 1.29 = 3.87 mm

Optional credit for a LOW-altitude installation (Table F.10):
      0 m -> factor 0.784 -> 2.352 mm
    500 m -> factor 0.833 -> 2.499 mm
   1000 m -> factor 0.884 -> 2.652 mm
   1500 m -> factor 0.942 -> 2.826 mm
   2000 m -> factor 1.000 -> 3.000 mm


## 16. From barriers to PCB design rules

The last step turns the per-barrier numbers into something you'd actually
type into a PCB tool's constraint manager: a domain-to-domain matrix of
minimum clearance and minimum creepage.

In [17]:
domain_names = [d.name for d in domains]
clearance_matrix = pd.DataFrame(index=domain_names, columns=domain_names, dtype=float)
creepage_matrix = pd.DataFrame(index=domain_names, columns=domain_names, dtype=float)

for _, row in all_results.iterrows():
    a, b = row["barrier"].split(" - ")
    clearance_matrix.loc[a, b] = clearance_matrix.loc[b, a] = row["clearance_mm_altitude_corrected"]
    creepage_matrix.loc[a, b] = creepage_matrix.loc[b, a] = row["creepage_mm"]

print("Minimum CLEARANCE (mm) between domains:")
display(clearance_matrix)
print()
print("Minimum CREEPAGE (mm) between domains:")
display(creepage_matrix)

Minimum CLEARANCE (mm) between domains:


,MAINS,DC_BUS,EARTH,COLD,MOTOR
MAINS,3.0,3.0,3.0,5.5,3.0
DC_BUS,3.0,0.5,3.0,5.5,0.5
EARTH,3.0,3.0,NaN,0.2,3.0
COLD,5.5,5.5,0.2,NaN,5.5
MOTOR,3.0,0.5,3.0,5.5,0.5



Minimum CREEPAGE (mm) between domains:


,MAINS,DC_BUS,EARTH,COLD,MOTOR
MAINS,2.80,2.80,2.8,6.24,3.12
DC_BUS,2.80,4.05,2.8,6.24,4.05
EARTH,2.80,2.80,NaN,0.40,2.80
COLD,6.24,6.24,0.4,NaN,6.24
MOTOR,3.12,4.05,2.8,6.24,2.80


## 17. Proving the modularity: a completely different board

The point of rebuilding the calculator around the full tables is that a
new design shouldn't need any new lookup tables -- just new instances of
the same `Environment` / `VoltageBlock` / `InsulationBarrier` classes. To
prove that out, here's an unrelated system: a small single-phase, 120 Vac,
**Class II** benchtop instrument (no protective earth -- reinforced
insulation does that job instead), pollution degree 2, standard FR-4
(material group IIIa), installed at 1500 m.

In [18]:
env_demo = Environment(pollution_degree=PollutionDegree.PD2, material_group=MaterialGroup.IIIa,
                        altitude_m=1500, on_printed_wiring=False)
calc_demo = IEC60664Calculator(env_demo, tables)

MAINS_120 = VoltageBlock("MAINS_120", "120 Vac single-phase input")
USER_PANEL = VoltageBlock("USER_PANEL", "User-accessible enclosure / knobs (Class II, no earth)")

demo_barriers = [
    InsulationBarrier(MAINS_120, USER_PANEL, InsulationType.BASIC, OvervoltageCategory.OVC_II,
                       working_voltage_rms_v=120, nominal_line_to_neutral_v=120),
    InsulationBarrier(MAINS_120, USER_PANEL, InsulationType.REINFORCED, OvervoltageCategory.OVC_II,
                       working_voltage_rms_v=120, nominal_line_to_neutral_v=120),
]

pd.DataFrame([calc_demo.evaluate(b) for b in demo_barriers])

,barrier,insulation,OVC,impulse_V,working_V_rms,clearance_mm,clearance_mm_altitude_corrected,creepage_mm,design_spacing_mm
0,MAINS_120 - USER_PANEL,basic,II,1500.0,120,0.5,0.5,0.8,0.8
1,MAINS_120 - USER_PANEL,reinforced,II,2500.0,120,1.5,1.5,1.6,1.6


Same classes, same tables, a completely different voltage/OVC/altitude
combination -- and the reinforced barrier's clearance and creepage both
come out roughly double the basic barrier's, as expected, with no changes
to `IEC60664Calculator` itself.

## 18. Summary and how to extend this further

Starting from the same system description as TI's SLUAAR5 white paper, this
notebook now:

1. parses the complete excerpted IEC 60664-1 Annex F tables (F.1, F.2, F.4,
   F.5, F.10) plus the CTI/pollution-degree/OVC reference tables directly
   out of `IsolationGuide.md`, instead of hardcoding a handful of points,
2. models the operating environment, voltage domains, and insulation
   barriers as small, reusable classes,
3. derives rated impulse voltage from a nominal mains voltage automatically
   (Table F.4 -> F.1, including the "one step higher" rule for reinforced
   insulation) alongside the option to supply a known impulse voltage
   directly,
4. re-derives the white paper's clearance numbers exactly, flags a specific,
   well-localized creepage discrepancy instead of papering over it (section
   13), and supports both installation-altitude corrections (Table A.2
   above 2000 m, Table F.10 below it),
5. produces the same domain-to-domain clearance/creepage matrix as before,
   and
6. demonstrates the whole pipeline on a second, unrelated board (section 17)
   with no changes to the calculator itself.

**To adapt this for a different design:**

- Add rows to `IsolationGuide.md` if you need more of Table F.1/F.2/F.5's
  range, or additional tables (F.3, F.6-F.9, Annex A's full altitude
  table) -- `parse_markdown_table()` will pick them up without any other
  code changes, since the tables are the single source of truth.
- Define a new `Environment`, domain list, and `InsulationBarrier` set for
  your own schematic, following section 17's pattern.
- For voltages/altitudes/material groups outside what's transcribed,
  extend the relevant table in `IsolationGuide.md` from the real standard
  rather than guessing at a value.

**Known caveats worth re-checking against the real standard before using
this for anything beyond a teaching exercise:**

| # | What | Where |
|---|---|---|
| 1 | Table F.2's blank cells are reconstructed via forward/backward-fill (a documented interpretation, not verified standard text) | Section 10 |
| 2 | Table F.5's PD2 "all groups except IIIb" vs. "material group II" column doesn't cleanly match the white paper's own worked creepage numbers | Section 13 |
| 3 | Table F.5's material-group ordering looks inverted relative to the usual "better CTI needs less creepage" convention | Section 13 |
| 4 | Table A.2 (installation above 2000 m) only has two reference points transcribed here | Section 15 |
| 5 | Parenthetical alternate values and footnote letters in the source tables (e.g. "8,0 (7,9) d") were simplified to the primary number during parsing | Section 10 |

Insulation coordination on a real product is safety-critical. Get the
complete standard tables and a design review from a qualified engineer
before any of this goes anywhere near a mains-connected board.

**Reference:** Chen Gao, "Circuit Board Insulation Design According to
IEC60664 for Motor Drive Application," Texas Instruments, SLUAAR5, August
2023.